In [24]:
!pip install -q torch_geometric
!pip install -q transformers accelerate bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.0/41.0 MB 21.7 MB/s eta 0:00:00


In [22]:
import ast
import math
import re
import random
import heapq
from dataclasses import dataclass, field
from typing import Union, Tuple, List, Optional, Dict, Any, Set
from collections import deque

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.distributions import Categorical
from torch_geometric.data import Data
from torch_geometric.nn import RGCNConv, global_mean_pool

import matplotlib.pyplot as plt

# ==========================================
# 1. AST NODE DEFINITIONS & PATTERN WILDCARD
# ==========================================

class ASTNode:
    def __lt__(self, other):
        return str(self) < str(other)

@dataclass(frozen=True)
class Num(ASTNode):
    value: Union[int, float]
    def __str__(self):
        return str(int(self.value)) if isinstance(self.value, float) and self.value.is_integer() else str(self.value)

@dataclass(frozen=True)
class Var(ASTNode):
    name: str
    def __str__(self): return self.name

@dataclass(frozen=True)
class PatternVar(ASTNode):
    name: str
    def __str__(self): return f"?{self.name}"

@dataclass(frozen=True)
class Op(ASTNode):
    op: str
    args: Tuple[ASTNode, ...]
    def __str__(self):
        if len(self.args) == 2 and self.op in ['+', '-', '*', '/', '**']:
            return f"({self.args[0]} {self.op} {self.args[1]})"
        return f"{self.op}({', '.join(str(a) for a in self.args)})"

@dataclass(frozen=True)
class ProofState(ASTNode):
    facts: Tuple[ASTNode, ...]  # Immutable set of active derived facts

    def __str__(self):
        return " & ".join(str(f) for f in self.facts)

# ==========================================
# 2. PARSER & EXTENDED VOCABULARY
# ==========================================

VOCAB = {
    '<UNK>': 0, '+': 1, '-': 2, '*': 3, '/': 4, '**': 5, 'Eq': 6,
    'NUM': 7, 'VAR': 8, 'sin': 9, 'cos': 10, 'tan': 11, 'log': 12,
    'ln': 13, 'sqrt': 14, 'exp': 15, 'dist': 16, 'area_circle': 17,
    'area_triangle': 18, 'angle_sum_tri': 19, 'abs': 20,

    # --- MATHEMATICAL LOGIC & SETS ---
    'And': 21, 'Or': 22, 'Implies': 23, 'Not': 24, 'Forall': 25, 'Exists': 26, 'In': 27,

    # --- NUMBER THEORY ---
    'mod': 28, 'gcd': 29, 'lcm': 30, 'divides': 31, 'is_prime': 32, 'congruent': 33,

    # --- COMBINATORICS ---
    'nCr': 34, 'nPr': 35, 'factorial': 36, 'summation': 37, 'product': 38,

    # --- GEOMETRY PREDICATES ---
    'parallel': 39, 'perp': 40, 'collinear': 41, 'congruent_tri': 42, 'similar_tri': 43,

    '&': 44,
    'pm': 45,  # "plus-or-minus", used by the quadratic formula (see rules below)
}

def parse_expr(expr_str: str) -> ASTNode:
    clean_str = re.sub(r'\?([a-zA-Z0-9_]+)', r'__pat_\1', expr_str)
    parsed = ast.parse(clean_str, mode='eval').body

    def _convert(node):
        if isinstance(node, ast.Constant):
            return Num(node.value)
        elif isinstance(node, ast.Name):
            if node.id.startswith('__pat_'):
                return PatternVar(node.id[6:])
            return Var(node.id)
        elif isinstance(node, ast.BinOp):
            op_map = {ast.Add: '+', ast.Sub: '-', ast.Mult: '*', ast.Div: '/', ast.Pow: '**', ast.BitAnd: '&'}
            return Op(op_map[type(node.op)], (_convert(node.left), _convert(node.right)))
        elif isinstance(node, ast.UnaryOp):
            if isinstance(node.op, ast.USub):
                return Op('*', (Num(-1), _convert(node.operand)))
            return _convert(node.operand)
        elif isinstance(node, ast.Call):
            return Op(node.func.id, tuple(_convert(a) for a in node.args))
        elif isinstance(node, ast.Compare) and len(node.ops) == 1 and isinstance(node.ops[0], ast.Eq):
            return Op('Eq', (_convert(node.left), _convert(node.comparators[0])))
        raise ValueError(f"Unsupported syntax: {type(node)}")

    return _convert(parsed)

# ==========================================
# 3. DOMAIN-EXPANDED RULE REGISTRY
# ==========================================

ALGEBRA_GEOMETRY_RULES = [
    # --- ALGEBRA 1 ATOMICS ---
    ("add_zero", "?A + 0 => ?A"),
    ("mul_one", "?A * 1 => ?A"),
    ("mul_zero", "?A * 0 => 0"),
    ("sub_self", "?A - ?A => 0"),
    ("div_self", "?A / ?A => 1", lambda b: b["A"] != Num(0)),
    ("mul_div_cancel", "(?A * ?B) / ?B => ?A", lambda b: b["B"] != Num(0)),
    ("div_mul_cancel", "(?A / ?B) * ?B => ?A"),
    ("distribute", "?A * (?B + ?C) <=> (?A * ?B) + (?A * ?C)"),
    ("pow_one", "?A ** 1 => ?A"),
    ("pow_zero", "?A ** 0 => 1"),
    ("distribute_sub", "?A * (?B - ?C) <=> (?A * ?B) - (?A * ?C)"),
    ("combine_sub_one", "(?A * ?X) - ?X => (?A - 1) * ?X"),
    ("combine_sub_same", "(?A * ?X) - (?B * ?X) => (?A - ?B) * ?X"),
    ("combine_add_same", "(?A * ?X) + (?B * ?X) => (?A + ?B) * ?X"),
    ("cancel_add_sub", "(?A + ?B) - ?B => ?A"),  # FIX: was missing, made add_var_both problems unsolvable
    ("cancel_sub_add", "(?A - ?B) + ?B => ?A"),  # FIX: symmetric counterpart
    # --- EQUALITY ISOLATION RULES (UNIDIRECTIONAL) ---
    ("eq_sub_both", "Eq(?A + ?B, ?C) <=> Eq(?A, ?C - ?B)"),
    ("eq_sub_var_right", "Eq(?A, ?B + ?C) <=> Eq(?A - ?B, ?C)"),
    ("eq_add_both", "Eq(?A - ?B, ?C) <=> Eq(?A, ?C + ?B)"),
    ("eq_div_both", "Eq(?A * ?B, ?C) <=> Eq(?A, ?C / ?B)"),
    ("eq_mul_both", "Eq(?A / ?B, ?C) <=> Eq(?A, ?C * ?B)"),
    ("eq_comm", "Eq(?A, ?B) <=> Eq(?B, ?A)"),
    (
        "cross_multiply",  # FIX: added — solving A/B = C/D needed this and had no path to it
        "Eq(?A / ?B, ?C / ?D) <=> Eq(?A * ?D, ?B * ?C)",
        lambda b: b["B"] != Num(0) and b["D"] != Num(0),
    ),
    # --- COMMUTATIVE RULES ---
    ("add_comm", "?A + ?B => ?B + ?A"),
    ("mul_comm", "?A * ?B => ?B * ?A"),
    # --- ALGEBRA 2: RADICALS & EXPONENTIALS ---
    ("sqrt_mul", "sqrt(?A * ?B) <=> sqrt(?A) * sqrt(?B)"),
    ("sqrt_div", "sqrt(?A / ?B) <=> sqrt(?A) / sqrt(?B)"),  # FIX: added — sibling of sqrt_mul, was missing
    ("sqrt_pow2", "sqrt(?A ** 2) => abs(?A)"),
    ("pow_half_sqrt", "?A ** 0.5 <=> sqrt(?A)"),
    ("exp_ln", "exp(ln(?A)) => ?A"),
    ("ln_e", "ln(e) => 1"),
    # FIX: exponent laws were entirely absent — added the three basic ones
    ("pow_mul_same_base", "(?A ** ?B) * (?A ** ?C) <=> ?A ** (?B + ?C)"),
    ("pow_div_same_base", "(?A ** ?B) / (?A ** ?C) <=> ?A ** (?B - ?C)"),
    ("pow_of_pow", "(?A ** ?B) ** ?C <=> ?A ** (?B * ?C)"),
    # --- ALGEBRA 2: LOGARITHMS ---
    ("log_mul", "log(?A * ?B) <=> log(?A) + log(?B)"),
    ("log_div", "log(?A / ?B) <=> log(?A) - log(?B)"),
    ("log_pow", "log(?A ** ?B) <=> ?B * log(?A)"),
    ("log_one", "log(1) => 0"),
    # --- ALGEBRA 2: COMPLEX NUMBERS & POLYNOMIALS ---
    ("i_square", "i ** 2 => -1"),
    ("sum_cubes", "?A**3 + ?B**3 <=> (?A + ?B) * ((?A**2 - (?A * ?B)) + ?B**2)"),
    (
        "diff_cubes",
        "?A**3 - ?B**3 <=> (?A - ?B) * ((?A**2 + (?A * ?B)) + ?B**2)",
    ),
    # FIX: factoring identities were missing — these are more fundamental
    # and more commonly needed than the cubes rules above
    ("diff_squares", "(?A**2) - (?B**2) <=> (?A - ?B) * (?A + ?B)"),
    ("perfect_square_add", "((?A**2) + (2 * (?A * ?B))) + (?B**2) <=> (?A + ?B) ** 2"),
    ("perfect_square_sub", "((?A**2) - (2 * (?A * ?B))) + (?B**2) <=> (?A - ?B) ** 2"),
    (
        "binomial_expand",  # FIX: general (A+B)(C+D) expansion, i.e. FOIL — had no general form
        "(?A + ?B) * (?C + ?D) <=> ((?A * ?C) + (?A * ?D)) + ((?B * ?C) + (?B * ?D))",
    ),
    # --- ALGEBRA 2: QUADRATIC EQUATIONS ---
    # FIX: the quadratic formula was entirely missing. Because it has two
    # roots, the RHS uses a "pm" (plus-or-minus) wrapper node around the
    # discriminant term rather than a single Num — is_solved_equation()
    # below has been extended to recognize that as a valid solved state.
    (
        "quadratic_formula",
        (
            "Eq(((?A * (?X ** 2)) + (?B * ?X)) + ?C, 0) <=> "
            "Eq(?X, (-?B + pm(sqrt((?B ** 2) - (4 * ?A * ?C)))) / (2 * ?A))"
        ),
        lambda b: b["A"] != Num(0),
    ),
    # FIX: monic variant (no explicit "1 *" coefficient on x**2) — matching
    # is purely structural, so "x**2 + B*x + C = 0" needs its own pattern,
    # it won't match the rule above written as-is.
    (
        "quadratic_formula_monic",
        (
            "Eq(((?X ** 2) + (?B * ?X)) + ?C, 0) <=> "
            "Eq(?X, (-?B + pm(sqrt((?B ** 2) - (4 * ?C)))) / 2)"
        ),
    ),
    # FIX: square-root property for the no-linear-term case ("x**2 = C" or,
    # after eq_add_both/eq_sub_both fire, "x**2 - C = 0" one step earlier)
    ("square_root_property", "Eq(?X ** 2, ?C) <=> Eq(?X, pm(sqrt(?C)))"),
    # --- GEOMETRY: TRIGONOMETRIC IDENTITIES ---
    ("pythagorean_trig", "(sin(?X)**2) + (cos(?X)**2) <=> 1"),
    ("tan_def", "tan(?X) <=> sin(?X) / cos(?X)"),
    # --- GEOMETRY: ANALYTIC & FORMULAS ---
    (
        "distance_formula",
        "dist(?x1, ?y1, ?x2, ?y2) <=> sqrt(((?x2 - ?x1)**2) + ((?y2 - ?y1)**2))",
    ),
    ("area_circle_def", "area_circle(?r) <=> pi * (?r**2)"),
    ("area_triangle_def", "area_triangle(?b, ?h) <=> 0.5 * ?b * ?h"),
    (
        "pythagoras_theorem",
        "Eq(?a**2 + ?b**2, ?c**2) <=> Eq(?c, sqrt(?a**2 + ?b**2))",
    ),
    ("triangle_angle_sum", "angle_sum_tri(?A, ?B, ?C) <=> Eq((?A + ?B) + ?C, 180)"),
    # --- NUMBER THEORY ---
    ("divides_transitive", "divides(?a, ?b) & divides(?b, ?c) => divides(?a, ?c)"),
    (
        "mod_congruence_add",
        (
            "congruent(?a, ?b, ?m) & congruent(?c, ?d, ?m) => congruent(?a +"
            " ?c, ?b + ?d, ?m)"
        ),
    ),
    ("gcd_lcm_identity", "gcd(?a, ?b) * lcm(?a, ?b) <=> ?a * ?b"),
    # --- COMBINATORICS ---
    ("pascal_identity", "nCr(?n, ?k) <=> nCr(?n - 1, ?k - 1) + nCr(?n - 1, ?k)"),
    ("handshake_lemma", "summation(?v, degree(?v)) <=> 2 * num_edges()"),
    # --- GEOMETRY LOGIC ---
    (
        "alternate_interior_angles",
        (
            "parallel(?L1, ?L2) & transversal(?T, ?L1, ?L2) => Eq(angle(?a),"
            " angle(?b))"
        ),
    ),
    (
        "perpendicular_transitive",
        "perp(?L1, ?L2) & perp(?L2, ?L3) => parallel(?L1, ?L3)",
    ),
]



def compile_rule_registry(rule_definitions):
    compiled_rules = {}
    rule_id = 0
    for rule in rule_definitions:
        name = rule[0]
        rule_str = rule[1]
        guard_fn = rule[2] if len(rule) > 2 else None

        if "<=>" in rule_str:
            lhs_str, rhs_str = rule_str.split("<=>", 1)
            lhs, rhs = parse_expr(lhs_str.strip()), parse_expr(rhs_str.strip())
            compiled_rules[rule_id] = (f"{name}_fwd", lhs, rhs, guard_fn)
            rule_id += 1
            compiled_rules[rule_id] = (f"{name}_rev", rhs, lhs, guard_fn)
            rule_id += 1
        elif "=>" in rule_str:
            lhs_str, rhs_str = rule_str.split("=>", 1)
            lhs, rhs = parse_expr(lhs_str.strip()), parse_expr(rhs_str.strip())
            compiled_rules[rule_id] = (name, lhs, rhs, guard_fn)
            rule_id += 1

    return compiled_rules

COMPILED_RULES = compile_rule_registry(ALGEBRA_GEOMETRY_RULES)
NUM_RULES = len(COMPILED_RULES)

# ==========================================
# 4. AST UTILITIES & SAFE PATH TRAVERSAL
# ==========================================

def get_ast_paths(node: ASTNode, current_path=()) -> List[Tuple[Tuple[int, ...], ASTNode]]:
    yield current_path, node
    if isinstance(node, Op):
        for i, arg in enumerate(node.args):
            yield from get_ast_paths(arg, current_path + (i,))

def replace_at_path(root: ASTNode, path: Tuple[int, ...], new_subnode: ASTNode) -> ASTNode:
    """Replaces node at path with safety check for out-of-bounds indices."""
    if not path:
        return new_subnode
    if not isinstance(root, Op):
        return root
    idx = path[0]
    if idx >= len(root.args):  # Guard against stale paths from mutated trees
        return root
    new_args = list(root.args)
    new_args[idx] = replace_at_path(root.args[idx], path[1:], new_subnode)
    return Op(root.op, tuple(new_args))

def evaluate_numeric_ops(node: ASTNode) -> Optional[ASTNode]:
    """Fold binary operations and common math functions on constant nodes."""
    if isinstance(node, Op):
        if len(node.args) == 2 and isinstance(node.args[0], Num) and isinstance(node.args[1], Num):
            v1, v2 = node.args[0].value, node.args[1].value
            if node.op == '+': return Num(v1 + v2)
            elif node.op == '-': return Num(v1 - v2)
            elif node.op == '*': return Num(v1 * v2)
            elif node.op == '/' and v2 != 0: return Num(v1 / v2)
            elif node.op == '**' and abs(v1 ** v2) < 1e6: return Num(v1 ** v2)
        elif len(node.args) == 1 and isinstance(node.args[0], Num):
            v = node.args[0].value
            if node.op == 'sqrt' and v >= 0: return Num(math.sqrt(v))
            elif node.op == 'log' and v > 0: return Num(math.log10(v))
            elif node.op == 'ln' and v > 0: return Num(math.log(v))
    return None

def match_pattern(pattern: ASTNode, expr: ASTNode, bindings: Dict[str, ASTNode]) -> Optional[Dict[str, ASTNode]]:
    if isinstance(pattern, PatternVar):
        if pattern.name in bindings:
            return bindings if bindings[pattern.name] == expr else None
        new_bindings = bindings.copy()
        new_bindings[pattern.name] = expr
        return new_bindings
    if type(pattern) != type(expr):
        return None
    if isinstance(pattern, (Num, Var)):
        return bindings if pattern == expr else None
    if isinstance(pattern, Op):
        if pattern.op != expr.op or len(pattern.args) != len(expr.args):
            return None
        curr_bindings = bindings
        for p_arg, e_arg in zip(pattern.args, expr.args):
            curr_bindings = match_pattern(p_arg, e_arg, curr_bindings)
            if curr_bindings is None:
                return None
        return curr_bindings
    return None

def substitute(pattern: ASTNode, bindings: Dict[str, ASTNode]) -> ASTNode:
    if isinstance(pattern, PatternVar):
        return bindings.get(pattern.name, Var(pattern.name))
    elif isinstance(pattern, Op):
        return Op(pattern.op, tuple(substitute(arg, bindings) for arg in pattern.args))
    return pattern

def extract_premises(pattern: ASTNode) -> List[ASTNode]:
    """Flattens chained '&' AST operations into a list of distinct premise patterns."""
    if isinstance(pattern, Op) and pattern.op == '&':
        return extract_premises(pattern.args[0]) + extract_premises(pattern.args[1])
    return [pattern]

def match_multi_premises(premises: List[ASTNode], facts: List[ASTNode], bindings: Dict[str, ASTNode] = None) -> List[Dict[str, ASTNode]]:
    """Recursively matches premise patterns against active facts while propagating variable bindings."""
    if bindings is None:
        bindings = {}
    if not premises:
        return [bindings]

    first_premise, remaining_premises = premises[0], premises[1:]
    matches = []

    for fact in facts:
        new_bindings = match_pattern(first_premise, fact, bindings)
        if new_bindings is not None:
            matches.extend(match_multi_premises(remaining_premises, facts, new_bindings))

    return matches

def apply_procedural_tactics(node: ASTNode) -> Optional[ASTNode]:
    changed = False
    current = node

    for _ in range(10):
        prev_str = str(current)

        # 1. Fold numeric constants
        applied_num = False
        for path, subnode in list(get_ast_paths(current)):
            eval_node = evaluate_numeric_ops(subnode)
            if eval_node:
                current = replace_at_path(current, path, eval_node)
                applied_num = True
                break
        if applied_num:
            changed = True
            continue

        # 2. Apply deterministic cleanup rules
        applied_rule = False
        for path, subnode in list(get_ast_paths(current)):
            # Unpack 4 items instead of 3
            for rule_id, (rule_name, lhs_pattern, rhs_pattern, guard_fn) in COMPILED_RULES.items():
                if rule_name in ["add_zero_fwd", "mul_one_fwd", "sub_self", "mul_zero", "div_self", "mul_div_cancel", "log_one"]:
                    bindings = match_pattern(lhs_pattern, subnode, {})
                    if bindings is not None:
                        # Evaluate guard function if present
                        if guard_fn is not None and not guard_fn(bindings):
                            continue

                        # Proceed with rewriting...
                        new_subnode = substitute(rhs_pattern, bindings)
                        current = replace_at_path(current, path, new_subnode)
                        applied_rule = True
                        break
            if applied_rule:
                break

        if applied_rule:
            changed = True
            continue

        if str(current) == prev_str:
            break

    return current if changed else None

def get_valid_transitions(root: ASTNode) -> List[Tuple[ASTNode, int, str]]:
    transitions = []
    root_str = str(root)

    tactic_node = apply_procedural_tactics(root)
    if tactic_node and str(tactic_node) != root_str:
        transitions.append((tactic_node, -1, "tactic_auto_cleanup"))

    for path, subnode in get_ast_paths(root):
        if isinstance(subnode, (Num, Var)):
            continue

        eval_node = evaluate_numeric_ops(subnode)
        if eval_node:
            new_root = replace_at_path(root, path, eval_node)
            if str(new_root) != root_str:
                transitions.append((new_root, -1, "eval_numeric"))

        for rule_id, rule_data in COMPILED_RULES.items():
            rule_name, lhs_pattern, rhs_pattern = rule_data[0], rule_data[1], rule_data[2]
            guard_fn = rule_data[3] if len(rule_data) > 3 else None

            bindings = match_pattern(lhs_pattern, subnode, {})
            if bindings is not None:
                # Evaluate guard condition if present (e.g., prevent division by zero)
                if guard_fn is not None and not guard_fn(bindings):
                    continue

                new_subnode = substitute(rhs_pattern, bindings)
                if new_subnode != subnode:
                    new_root = replace_at_path(root, path, new_subnode)
                    if str(new_root) != root_str:
                        transitions.append((new_root, rule_id, rule_name))

    return transitions

def get_valid_proof_transitions(state: ProofState) -> List[Tuple[ProofState, int, str]]:
    """Derives new facts by matching multi-premise rules against the active fact set."""
    transitions = []
    existing_facts = set(state.facts)

    for rule_id, rule_data in COMPILED_RULES.items():
        rule_name, lhs_pattern, rhs_pattern = rule_data[0], rule_data[1], rule_data[2]
        guard_fn = rule_data[3] if len(rule_data) > 3 else None

        premises = extract_premises(lhs_pattern)
        match_results = match_multi_premises(premises, list(state.facts))

        for bindings in match_results:
            # Check guard function against multi-fact bindings
            if guard_fn is not None and not guard_fn(bindings):
                continue

            derived_fact = substitute(rhs_pattern, bindings)
            if derived_fact not in existing_facts:
                new_state = ProofState(facts=state.facts + (derived_fact,))
                transitions.append((new_state, rule_id, rule_name))

    return transitions

# ==========================================
# 5. NEURAL MODEL & ENVIRONMENT
# ==========================================

def get_node_type_id(node: ASTNode) -> int:
    if isinstance(node, Num): return VOCAB['NUM']
    elif isinstance(node, Var): return VOCAB['VAR']
    elif isinstance(node, Op): return VOCAB.get(node.op, VOCAB['<UNK>'])
    return VOCAB['<UNK>']

EDGE_TYPES = {
    'AST_CHILD': 0,
    'EQUAL_TO': 1,
    'PERPENDICULAR': 2,
    'PERP': 2,
    'PARALLEL': 3,
    'DIVIDES': 4,
    'LOGICAL_AND': 5
}

def proof_state_to_relational_graph(facts: List[ASTNode]) -> Data:
    """Converts a collection of logical facts/ASTs into a single multi-relational Knowledge Graph."""
    nodes, edges, edge_types = [], [], []

    def _traverse_ast(node: ASTNode, parent_idx: int = None):
        curr_idx = len(nodes)
        nodes.append(get_node_type_id(node))

        if parent_idx is not None:
            edges.append((parent_idx, curr_idx))
            edge_types.append(EDGE_TYPES['AST_CHILD'])

        if isinstance(node, Op):
            if node.op in ['perp', 'parallel', 'divides'] and len(node.args) == 2:
                rel_type = EDGE_TYPES.get(node.op.upper(), EDGE_TYPES['AST_CHILD'])
                idx1 = _traverse_ast(node.args[0], curr_idx)
                idx2 = _traverse_ast(node.args[1], curr_idx)
                edges.extend([(idx1, idx2), (idx2, idx1)])
                edge_types.extend([rel_type, rel_type])
            else:
                for child in node.args:
                    _traverse_ast(child, curr_idx)
        return curr_idx

    for fact in facts:
        _traverse_ast(fact)

    x = torch.tensor(nodes, dtype=torch.long).unsqueeze(-1)
    edge_index = torch.tensor(edges, dtype=torch.long).t().contiguous() if edges else torch.empty((2, 0), dtype=torch.long)
    edge_type = torch.tensor(edge_types, dtype=torch.long)

    return Data(x=x, edge_index=edge_index, edge_type=edge_type)

class RLUniversalMathRGNN(nn.Module):
    def __init__(self, vocab_size=64, embed_dim=64, hidden_dim=128, num_relations=len(EDGE_TYPES), num_rules=NUM_RULES):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim)

        self.conv1 = RGCNConv(embed_dim, hidden_dim, num_relations=num_relations)
        self.conv2 = RGCNConv(hidden_dim, hidden_dim, num_relations=num_relations)

        self.actor = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim), nn.ReLU(), nn.Linear(hidden_dim, num_rules)
        )
        self.critic = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim), nn.ReLU(), nn.Linear(hidden_dim, 1)
        )

    def forward(self, data: Data, action_mask: torch.Tensor = None):
        x, edge_index, edge_type, batch = data.x.squeeze(-1), data.edge_index, data.edge_type, data.batch

        h = F.relu(self.embedding(x))
        h = F.relu(self.conv1(h, edge_index, edge_type))
        h = F.relu(self.conv2(h, edge_index, edge_type))

        pooled = global_mean_pool(h, batch)
        logits = self.actor(pooled)
        value = self.critic(pooled)

        if action_mask is not None:
            mask_penalty = (1.0 - action_mask.float()) * -1e9
            logits = logits + mask_penalty

        return logits, value.squeeze(-1)

# ==========================================
# 6. ADAPTIVE SEARCH SOLVER
# ==========================================

def evaluate_ast_complexity(node: ASTNode) -> int:
    count = 1
    if isinstance(node, Op):
        for arg in node.args:
            count += evaluate_ast_complexity(arg)
    return count

def _contains_var(node: ASTNode) -> bool:
  """True if node still has a free variable anywhere in it."""
  if isinstance(node, Var):
    return True
  if isinstance(node, Op):
    return any(_contains_var(a) for a in node.args)
  return False

def _contains_pm(node: ASTNode) -> bool:
  """True if node contains a pm(...) ("plus-or-minus") wrapper anywhere."""
  if isinstance(node, Op):
    if node.op == "pm":
      return True
    return any(_contains_pm(a) for a in node.args)
  return False

def _is_fully_folded(node: ASTNode) -> bool:
  """True if no further numeric constant-folding is possible anywhere in node."""
  return all(evaluate_numeric_ops(sub) is None for _, sub in get_ast_paths(node))

def is_solved_equation(node: ASTNode) -> bool:
  """Returns True if the AST represents an isolated variable (e.g., x = 10 or 10 = x).

  Also accepts "x = <fully-folded numeric expression containing pm(...)>" as
  solved — this is the natural terminal form produced by the quadratic
  formula, which has two roots and can't be collapsed to a single Num. It
  must still be fully numerically folded (no leftover "3 - 1"-style
  subexpressions) to count, for consistency with the plain-Num case above.
  """
  if isinstance(node, Op) and node.op == "Eq":
    lhs, rhs = node.args
    if isinstance(lhs, Var) and isinstance(rhs, Num):
      return True
    if isinstance(rhs, Var) and isinstance(lhs, Num):
      return True
    # FIX: quadratic formula's answer form — no variable left, but a pm(...)
    # marker instead of a single Num.
    if (
        isinstance(lhs, Var)
        and not _contains_var(rhs)
        and _contains_pm(rhs)
        and _is_fully_folded(rhs)
    ):
      return True
  return False


def solve_adaptive(
    model,
    start_ast: ASTNode,
    target_ast: ASTNode = None,
    task_mode: str = "AUTO",
    max_expansions=1000,
):
  model.eval()
  device = next(model.parameters()).device  # Dynamically detect model device

  # Auto mode routing
  if task_mode == "AUTO":
    if target_ast is not None:
      task_mode = "PROVE"
    elif isinstance(start_ast, Op) and start_ast.op == "Eq":
      task_mode = "SOLVE"
    else:
      task_mode = "SIMPLIFY"

  open_set = []
  heapq.heappush(open_set, (0, 0, start_ast, []))
  closed = set()
  best_ast, min_complexity = start_ast, evaluate_ast_complexity(start_ast)
  best_path, expansions = [], 0

  while open_set and expansions < max_expansions:
    f_score, g_cost, current_ast, path = heapq.heappop(open_set)
    ast_str = str(current_ast)

    if ast_str in closed:
      continue
    closed.add(ast_str)
    expansions += 1

    curr_comp = evaluate_ast_complexity(current_ast)
    if curr_comp < min_complexity:
      min_complexity, best_ast, best_path = curr_comp, current_ast, path

    # TERMINATION CHECK 1: PROVE MODE
    if task_mode == "PROVE" and target_ast:
      if (
          isinstance(current_ast, ProofState)
          and target_ast in current_ast.facts
      ):
        return path + [(current_ast, "TARGET_REACHED")], expansions, "PROVED"
      elif ast_str == str(target_ast):
        return path + [(current_ast, "TARGET_REACHED")], expansions, "PROVED"

    # TERMINATION CHECK 2: SOLVE MODE
    if task_mode == "SOLVE" and is_solved_equation(current_ast):
      return path + [(current_ast, "VARIABLE_ISOLATED")], expansions, "SOLVED"

    # Expand state transitions
    if isinstance(current_ast, ProofState):
      transitions = get_valid_proof_transitions(current_ast)
      graph = proof_state_to_relational_graph(list(current_ast.facts))
    else:
      transitions = get_valid_transitions(current_ast)
      graph = proof_state_to_relational_graph([current_ast])

    # Move graph to the matching device (CUDA/CPU)
    graph = graph.to(device)
    graph.batch = torch.zeros(graph.x.size(0), dtype=torch.long, device=device)
    if not transitions:
      continue

    mask = torch.zeros(NUM_RULES, dtype=torch.bool, device=device)
    for _, r_id, _ in transitions:
      if r_id >= 0:
        mask[r_id] = True

    with torch.no_grad():
      logits, value = model(graph, action_mask=mask)
      probs = torch.softmax(logits, dim=-1)

    for next_ast, r_id, r_name in transitions:
      prob = probs[0, r_id].item() if r_id >= 0 else 0.5
      new_g = g_cost + (1.0 - 0.8 * prob)
      v_clamped = max(-1.0, min(1.0, value.item()))
      h = (1.0 - v_clamped) * 5.0
      f = new_g + h
      heapq.heappush(open_set, (f, new_g, next_ast, path + [(next_ast, r_name)]))

  if task_mode == "SIMPLIFY":
    return (
        best_path + [(best_ast, "MIN_COMPLEXITY_FOUND")],
        expansions,
        "SIMPLIFIED",
    )

  return None, expansions, "FAILED"

# ==========================================
# 6.5 HARD PROBLEM GENERATORS
# ==========================================


def get_inverse_rule_id(rule_id: int, compiled_rules: dict) -> Optional[int]:
    """
    Finds the inverse rule ID for a given rule_id in COMPILED_RULES.
    Returns None if the rule is destructive/non-invertible (e.g. mul_zero: ?A * 0 => 0).
    """
    if rule_id not in compiled_rules:
        return None

    rule_data = compiled_rules[rule_id]
    rule_name, lhs, rhs = rule_data[0], rule_data[1], rule_data[2]

    # 1. Match bidirectional rule pairs generated by <=> (<name>_fwd / <name>_rev)
    if rule_name.endswith("_fwd"):
        target_name = rule_name[:-4] + "_rev"
        for r_id, r_data in compiled_rules.items():
            if r_data[0] == target_name:
                return r_id
    elif rule_name.endswith("_rev"):
        target_name = rule_name[:-4] + "_fwd"
        for r_id, r_data in compiled_rules.items():
            if r_data[0] == target_name:
                return r_id

    # 2. Structural matching: find a rule where LHS == RHS and RHS == LHS
    str_lhs, str_rhs = str(lhs), str(rhs)
    for r_id, r_data in compiled_rules.items():
        if r_id == rule_id:
            continue
        other_lhs, other_rhs = r_data[1], r_data[2]
        if str(other_lhs) == str_rhs and str(other_rhs) == str_lhs:
            return r_id

    # Irreversible transformation (e.g. x * 0 => 0 cannot be uniquely inverted)
    return None

INVERSE_RULE_MAP = {
    rule_id: get_inverse_rule_id(rule_id, COMPILED_RULES)
    for rule_id in COMPILED_RULES
}

def generate_supervised_algebra_trajectory(depth: int = 6):
    val = random.randint(1, 20)
    solved_target = Op('Eq', (Var('x'), Num(val)))

    current = solved_target
    forward_trajectory = [] # Stores tuples of (ASTNode, target_rule_id)

    for _ in range(depth):
        transitions = get_valid_transitions(current)
        if not transitions:
            break

        # AFTER
        next_ast, r_id, _ = random.choice(transitions)

        # Ensure r_id is an int if passed as a Rule object
        rule_key = getattr(r_id, "rule_id", getattr(r_id, "r_id", r_id))
        inv_r_id = INVERSE_RULE_MAP.get(rule_key)

        # Only accept steps that have a valid executable inverse rule
        if inv_r_id is not None:
            # The next_ast in backward walk is the PREVIOUS state in forward solution
            forward_trajectory.append((next_ast, inv_r_id))
            current = next_ast

    if not forward_trajectory:
        return None

    # Reverse to order from Start Problem -> Solved Target
    forward_trajectory.reverse()
    start_problem = forward_trajectory[0][0]

    # Final step transition goes to solved_target
    return start_problem, solved_target, forward_trajectory

def generate_hard_algebra_problem(depth: int = 1):
    val = random.randint(1, 15)
    target = Op('Eq', (Var('x'), Num(val)))

    # Retrieve compiled rule IDs for teacher trajectory synthesis
    rule_name_to_id = {data[0]: r_id for r_id, data in COMPILED_RULES.items()}

    states = [target]
    forward_rules = []
    current_state = target

    # Build problem backward from target to create branching search trees
    for step in range(depth):
        op_choices = ['add_const', 'sub_const', 'mul_const', 'add_var_both']
        choice = random.choice(op_choices)

        lhs, rhs = current_state.args[0], current_state.args[1]

        if choice == 'add_const':
            k = random.randint(1, 9)
            new_lhs = Op('+', (lhs, Num(k)))
            r_id = rule_name_to_id.get("eq_sub_both_fwd", -1)  # Added _fwd
            new_rhs = Num(rhs.value + k) if isinstance(rhs, Num) else Op('+', (rhs, Num(k)))

        elif choice == 'sub_const':
            k = random.randint(1, 9)
            new_lhs = Op('-', (lhs, Num(k)))
            r_id = rule_name_to_id.get("eq_add_both_fwd", -1)  # Added _fwd
            new_rhs = Num(rhs.value - k) if isinstance(rhs, Num) else Op('-', (rhs, Num(k)))

        elif choice == 'mul_const':
            k = random.randint(2, 5)
            new_lhs = Op('*', (lhs, Num(k)))
            r_id = rule_name_to_id.get("eq_div_both_fwd", -1)  # Added _fwd
            new_rhs = Num(rhs.value * k) if isinstance(rhs, Num) else Op('*', (rhs, Num(k)))

        elif choice == 'add_var_both':
            k = random.randint(1, 3)
            var_term = Op('*', (Num(k), Var('x'))) if k > 1 else Var('x')
            new_lhs = Op('+', (lhs, var_term))
            r_id = rule_name_to_id.get("eq_sub_var_right_fwd", -1)  # Added _fwd
            new_rhs = Op('+', (var_term, rhs)) if isinstance(rhs, Num) else Op('+', (rhs, var_term))

        current_state = Op('Eq', (new_lhs, new_rhs))
        forward_rules.append(r_id)
        states.append(current_state)

    start_ast = states[-1]

    # Construct exact forward solution trajectory (start -> target)
    synth_trajectory = []
    for i in range(len(forward_rules) - 1, -1, -1):
        synth_trajectory.append((states[i + 1], forward_rules[i]))

    return start_ast, target, synth_trajectory


def generate_hard_proof_problem():
    fact1 = parse_expr("parallel(L1, L2)")
    fact2 = parse_expr("transversal(T, L1, L2)")
    start_state = ProofState(facts=(fact1, fact2))
    target_ast = parse_expr("Eq(angle(a), angle(b))")
    return start_state, target_ast


# ==========================================
# 7. RL STATE TRANSITION & TERMINAL UTILS
# ==========================================

def get_transitions(state: Union[ASTNode, ProofState]):
    if isinstance(state, ProofState):
        return get_valid_proof_transitions(state)
    return get_valid_transitions(state)


def is_terminal(state: Union[ASTNode, ProofState], target_ast: Optional[ASTNode], task_mode: str) -> bool:
    if task_mode == "PROVE" and target_ast:
        if isinstance(state, ProofState) and target_ast in state.facts:
            return True
        if str(state) == str(target_ast):
            return True
    elif task_mode == "SOLVE":
        if target_ast and str(state) == str(target_ast):
            return True
        if is_solved_equation(state):
            return True
    return False

# ==========================================
# 8. CURRICULUM LEARNING CONTROLLER
# ==========================================

class CurriculumManager:
    def __init__(self, min_depth: int = 2, max_depth: int = 15, window_size: int = 20):
        self.current_depth = min_depth
        self.max_depth = max_depth
        self.history = deque(maxlen=window_size)

    def record_result(self, success: bool):
        self.history.append(1.0 if success else 0.0)
        if len(self.history) == self.history.maxlen:
            success_rate = sum(self.history) / len(self.history)
            if success_rate >= 0.85 and self.current_depth < self.max_depth:
                self.current_depth += 1
                self.history.clear()
                print(f"\n[CURRICULUM UPGRADE] Promoted to Complexity Depth Tier: {self.current_depth}!\n")

    @property
    def success_rate(self) -> float:
        return (sum(self.history) / len(self.history)) if self.history else 0.0

@dataclass
class FailedProblem:
    start_ast: Any
    target_ast: Any
    task_mode: str
    max_expansions: int
    noise: float = 0.1
    attempts: int = 1
    synth_trajectory: Optional[List] = field(default_factory=list)


class FailureReplayBuffer:

    def __init__(self, capacity: int = 200, noise_step: float = 0.15):
        self.buffer: List[FailedProblem] = []
        self.capacity = capacity
        self.noise_step = noise_step

    def add(
        self,
        start_ast: Any,
        target_ast: Any,
        task_mode: str,
        base_expansions: int,
        synth_trajectory: Optional[List] = None,
    ):
        if len(self.buffer) >= self.capacity:
            self.buffer.pop(0)

        failed_item = FailedProblem(
            start_ast=start_ast,
            target_ast=target_ast,
            task_mode=task_mode,
            max_expansions=int(base_expansions * 1.5),
            noise=self.noise_step,
            attempts=1,
            synth_trajectory=synth_trajectory or [],
        )
        self.buffer.append(failed_item)

    def sample(self) -> FailedProblem:
        idx = random.randint(0, len(self.buffer) - 1)
        return self.buffer.pop(idx)

    def re_add(self, item: FailedProblem):
        item.max_expansions = min(int(item.max_expansions * 1.5), 400)  # FIX: cap runaway budget growth
        item.noise += self.noise_step
        item.attempts += 1

        if item.attempts > 8:  # FIX: evict likely-unsolvable "zombie" items instead of keeping them forever
            return

        if len(self.buffer) >= self.capacity:
            self.buffer.pop(0)
        self.buffer.append(item)

    def __len__(self):
        return len(self.buffer)

# ==========================================
# 9. MATPLOTLIB VISUALIZATION DASHBOARD
# ==========================================

class TrainingVisualizer:
    # FIX: the old version used plt.ion()/fig.canvas.draw()/plt.pause() to try
    # to animate the plot every epoch. Colab's default renderer doesn't
    # redraw a figure in place like a local Jupyter/Qt backend does, so this
    # was never actually updating live there — it also added real per-epoch
    # overhead (a draw + pause call 1000 times) for no visible benefit.
    # Now log() just records the numbers, and the figure is built once, at
    # the end, from the full history.

    def __init__(self):
        self.epochs, self.losses, self.success_rates, self.depths = [], [], [], []

    def log(self, epoch: int, loss: float, success_rate: float, depth: int):
        self.epochs.append(epoch)
        self.losses.append(loss)
        self.success_rates.append(success_rate)
        self.depths.append(depth)

    def keep_open(self):
        fig, axs = plt.subplots(3, 1, figsize=(9, 8), sharex=True)
        fig.suptitle('Neural Symbolic RL Engine - Training Summary', fontsize=12, fontweight='bold')

        axs[0].plot(self.epochs, self.losses, color='#e74c3c', linewidth=2, label='Loss')
        axs[0].set_ylabel('Loss')
        axs[0].legend(loc='upper right')

        axs[1].plot(self.epochs, self.success_rates, color='#2ecc71', linewidth=2, label='Win Rate')
        axs[1].axhline(y=0.85, color='gray', linestyle='--', alpha=0.7)
        axs[1].set_ylabel('Win Rate')
        axs[1].set_ylim(-0.05, 1.05)
        axs[1].legend(loc='lower right')

        axs[2].plot(self.epochs, self.depths, color='#3498db', linewidth=2, drawstyle='steps-post', label='Depth')
        axs[2].set_ylabel('Depth')
        axs[2].set_xlabel('Epochs')
        axs[2].legend(loc='upper left')

        plt.tight_layout()
        plt.show()

# ==========================================
# 10. SEARCH-GUIDED CURRICULUM & EXPERT ITERATION
# ==========================================

@dataclass(order=True)
class PrioritizedNode:
    priority: float
    state: Any = field(compare=False)
    path: List[Tuple[Any, int]] = field(default_factory=list, compare=False)
    log_prob_accum: float = field(default=0.0, compare=False)
    depth: int = field(default=0, compare=False)


def neural_best_first_search(
    model: RLUniversalMathRGNN,
    start_state: Any,
    target_ast: Any,
    task_mode: str,
    max_expansions: int = 150,
    depth_penalty: float = 0.05,
    depth_penalty_window: int = 3,  # FIX: no penalty for the first N steps (see below)
    exploration_noise: float = 0.125,  # Escalating noise factor
) -> Optional[List[Tuple[Any, int]]]:
    model.eval()
    device = next(model.parameters()).device

    frontier: List[PrioritizedNode] = []
    visited: Set[str] = set()

    def get_state_key(st):
        return str(st.facts) if hasattr(st, "facts") else str(st)

    visited.add(get_state_key(start_state))
    initial_node = PrioritizedNode(
        priority=0.0,
        state=start_state,
        path=[],
        log_prob_accum=0.0,
        depth=0,
    )
    heapq.heappush(frontier, initial_node)

    expansions = 0

    while frontier and expansions < max_expansions:
        current_node: PrioritizedNode = heapq.heappop(frontier)
        current_state = current_node.state
        expansions += 1

        if is_terminal(current_state, target_ast, task_mode):
            return current_node.path

        transitions = get_transitions(current_state)
        if not transitions:
            continue

        facts = (
            list(current_state.facts)
            if isinstance(current_state, ProofState)
            else [current_state]
        )
        graph = proof_state_to_relational_graph(facts).to(device)
        # ADD device=device HERE
        graph.batch = torch.zeros(graph.x.size(0), dtype=torch.long, device=device)

        # ADD device=device HERE
        mask = torch.zeros(NUM_RULES, dtype=torch.bool, device=device)
        for _, r_id, _ in transitions:
            if r_id >= 0:
                mask[r_id] = True

        with torch.no_grad():
            logits, val_pred = model(graph, action_mask=mask)
            logits = logits.squeeze(0)

            if exploration_noise > 0.0:
                logits = logits + (torch.randn_like(logits) * exploration_noise)

            val_pred = val_pred.squeeze()
            masked_logits = torch.where(
                mask, logits, torch.tensor(-1e9, device=logits.device)
            )
            log_probs = F.log_softmax(masked_logits, dim=-1)
            predicted_value = torch.tanh(val_pred).item()

        for next_state, r_id, r_name in transitions:
            child_key = get_state_key(next_state)
            if child_key in visited:
                continue
            visited.add(child_key)

            rule_log_prob = log_probs[r_id].item() if r_id >= 0 else 0.0
            new_log_prob_accum = current_node.log_prob_accum + rule_log_prob
            new_depth = current_node.depth + 1

            # FIX: give the search a free "exploration window" of
            # depth_penalty_window steps before the depth penalty kicks in.
            # Previously every additional hop was penalized from step 1,
            # which biased the frontier toward shallow dead-ends instead of
            # continuing to expand toward an actual solution — the model
            # would settle into a local minimum a couple of steps deep
            # rather than pushing further when the real answer needed more
            # steps than that.
            penalized_depth = max(0, new_depth - depth_penalty_window)
            score = (
                predicted_value
                + (0.1 * new_log_prob_accum)
                - (depth_penalty * penalized_depth)
            )
            new_path = current_node.path + [(current_state, r_id)]

            child_node = PrioritizedNode(
                priority=-score,
                state=next_state,
                path=new_path,
                log_prob_accum=new_log_prob_accum,
                depth=new_depth,
            )
            heapq.heappush(frontier, child_node)

    return None

def compute_trajectory_loss(
    model: RLUniversalMathRGNN,
    trajectory: List[Any],
    device: torch.device,
    gamma: float = 0.95,
) -> torch.Tensor:
    """Computes combined policy Cross-Entropy and value MSE loss across a trajectory."""
    total_policy_loss = torch.tensor(0.0, device=device)
    total_value_loss = torch.tensor(0.0, device=device)
    valid_steps = 0
    path_len = len(trajectory)

    for t, item in enumerate(trajectory):
        state = None
        target_rule_id = -1

        # Handle tuple steps, state objects, or raw Op AST nodes
        if isinstance(item, (tuple, list)):
            state = item[0]
            raw_rule = item[1] if len(item) > 1 else -1
            if isinstance(raw_rule, int):
                target_rule_id = raw_rule
            elif hasattr(raw_rule, "rule_id"):
                target_rule_id = getattr(raw_rule, "rule_id")
            elif hasattr(raw_rule, "r_id"):
                target_rule_id = getattr(raw_rule, "r_id")
        elif hasattr(item, "rule_id") or hasattr(item, "r_id"):
            state = getattr(item, "state", item)
            target_rule_id = getattr(item, "rule_id", getattr(item, "r_id", -1))
        else:
            state = item

        # Infer the target rule if only raw AST states are present in the trajectory
        if target_rule_id < 0 and t + 1 < len(trajectory):
            next_item = trajectory[t + 1]
            next_state_target = (
                next_item[0]
                if isinstance(next_item, (tuple, list))
                else getattr(next_item, "state", next_item)
            )

            target_key = (
                str(next_state_target.facts)
                if hasattr(next_state_target, "facts")
                else str(next_state_target)
            )
            transitions = get_transitions(state)
            for next_st, r_id, *_ in transitions:
                st_key = (
                    str(next_st.facts)
                    if hasattr(next_st, "facts")
                    else str(next_st)
                )
                if st_key == target_key:
                    target_rule_id = r_id
                    break

        if state is None:
            continue

        transitions = get_transitions(state)
        if not transitions:
            continue

        mask = torch.zeros(NUM_RULES, dtype=torch.bool, device=device)
        for next_st, r_id, *_ in transitions:
            if r_id >= 0:
                mask[r_id] = True

        if target_rule_id < 0 or target_rule_id >= NUM_RULES:
            continue

        # Force mask to True for supervised target so CrossEntropy can calculate loss
        mask[target_rule_id] = True

        facts = list(state.facts) if isinstance(state, ProofState) else [state]
        graph = proof_state_to_relational_graph(facts).to(device)
        graph.batch = torch.zeros(
            graph.x.size(0), dtype=torch.long, device=device
        )

        logits, val_pred = model(graph, action_mask=mask)
        masked_logits = logits.view(1, -1).masked_fill(~mask.view(1, -1), -1e9)

        target_tensor = torch.tensor(
            [target_rule_id], dtype=torch.long, device=device
        )
        p_loss = F.cross_entropy(masked_logits, target_tensor)

        steps_remaining = path_len - 1 - t
        target_val = torch.tensor(
            [gamma**steps_remaining], dtype=torch.float, device=device
        )
        v_loss = F.mse_loss(torch.tanh(val_pred).view(-1), target_val)

        total_policy_loss += p_loss
        total_value_loss += v_loss
        valid_steps += 1

    if valid_steps > 0:
        return (total_policy_loss / valid_steps) + 0.5 * (
            total_value_loss / valid_steps
        )
    return torch.tensor(0.0, device=device)


def train_rl_curriculum(
    model: RLUniversalMathRGNN, epochs: int = 200
) -> TrainingVisualizer:
    optimizer = torch.optim.AdamW(
        model.parameters(), lr=1e-3, weight_decay=1e-4
    )
    curriculum = CurriculumManager(min_depth=1, max_depth=8)
    visualizer = TrainingVisualizer()
    failure_buffer = FailureReplayBuffer(capacity=100, noise_step=0.15)
    device = next(model.parameters()).device

    print(
        f"=== Starting Neural Search with Failure Replay Buffer ({epochs} Epochs) ==="
    )

    for epoch in range(1, epochs + 1):
        sampled_from_buffer = False
        current_failed_item = None
        synth_trajectory = []

        # 1. Decide whether to replay a failure or generate a new problem
        if len(failure_buffer) > 0 and random.random() < 0.4:
            current_failed_item = failure_buffer.sample()
            start_ast = current_failed_item.start_ast
            target_ast = current_failed_item.target_ast
            task_mode = current_failed_item.task_mode
            max_exp = current_failed_item.max_expansions
            noise = current_failed_item.noise
            # RETRIEVE STORED TEACHER TRAJECTORY FROM BUFFER
            synth_trajectory = getattr(current_failed_item, "synth_trajectory", [])
            sampled_from_buffer = True
        else:
            if random.random() < 0.75:
                start_ast, target_ast, synth_trajectory = generate_hard_algebra_problem(
                    depth=curriculum.current_depth
                )
                task_mode = "SOLVE"
            else:
                start_ast, target_ast = generate_hard_proof_problem()
                task_mode = "PROVE"
            # INCREASED SEARCH BUDGET FOR HIGHER DEPTHS
            max_exp = 60 + (curriculum.current_depth * 35)
            noise = 0.0

        # 2. Search Rollout
        winning_path = neural_best_first_search(
            model=model,
            start_state=start_ast,
            target_ast=target_ast,
            task_mode=task_mode,
            max_expansions=max_exp,
            exploration_noise=noise,
        )

        solved = winning_path is not None
        curriculum.record_result(solved)

        # 3. Handle Failure Buffer State Updates
        if not solved:
            if sampled_from_buffer and current_failed_item:
                failure_buffer.re_add(current_failed_item)
            else:
                failure_buffer.add(
                    start_ast=start_ast,
                    target_ast=target_ast,
                    task_mode=task_mode,
                    base_expansions=max_exp,
                    synth_trajectory=synth_trajectory
                )

        # 4. Gradient Updates (Dual Path)
        loss_val = 0.0
        model.train()
        optimizer.zero_grad()

        if solved and winning_path and len(winning_path) > 0:
            # Path A: Search succeeded -> Learn directly from winning search path
            loss = compute_trajectory_loss(model, winning_path, device)
        elif synth_trajectory and len(synth_trajectory) > 0:
            # Path B: Search failed -> Bootstrap model using generator's synthetic steps
            loss = compute_trajectory_loss(model, synth_trajectory, device)
        else:
            loss = torch.tensor(0.0, device=device)

        if loss.item() > 0.0:
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            loss_val = loss.item()

        # 5. Logging
        visualizer.log(
            epoch, loss_val, curriculum.success_rate, curriculum.current_depth
        )
        path_length = len(winning_path) if winning_path else 0

        if epoch % 5 == 0 or epoch == epochs:
            print(
                f"Epoch {epoch:03d}/{epochs} | Depth: {curriculum.current_depth} |"
                f" Steps: {path_length:02d} | WinRate:"
                f" {curriculum.success_rate * 100:.1f}% | Loss: {loss_val:.4f} |"
                f" BufferSize: {len(failure_buffer)} | Solved: {solved}"
            )

    return visualizer

# ==========================================
# 11. MAIN EXECUTION
# ==========================================

if __name__ == "__main__":
    # 1. Detect and set device
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    # 2. Move model weights to CUDA
    model = RLUniversalMathRGNN(vocab_size=len(VOCAB) + 10, num_rules=NUM_RULES).to(device)

    # 3. Train
    visualizer = train_rl_curriculum(model, epochs=400)

    # 4. Hold visualizer window open at the end
    visualizer.keep_open()

  # Save only the learned weights (recommended)
    torch.save(model.state_dict(), "math_rgnn_weights.pth")

=== Starting Neural Search with Failure Replay Buffer (400 Epochs) ===
Epoch 005/400 | Depth: 1 | Steps: 02 | WinRate: 100.0% | Loss: 2.0300 | BufferSize: 0 | Solved: True
Epoch 010/400 | Depth: 1 | Steps: 02 | WinRate: 100.0% | Loss: 1.0342 | BufferSize: 0 | Solved: True
Epoch 015/400 | Depth: 1 | Steps: 01 | WinRate: 100.0% | Loss: 0.0001 | BufferSize: 0 | Solved: True

[CURRICULUM UPGRADE] Promoted to Complexity Depth Tier: 2!

Epoch 020/400 | Depth: 2 | Steps: 02 | WinRate: 0.0% | Loss: 1.2083 | BufferSize: 0 | Solved: True
Epoch 025/400 | Depth: 2 | Steps: 01 | WinRate: 100.0% | Loss: 0.0000 | BufferSize: 0 | Solved: True
Epoch 030/400 | Depth: 2 | Steps: 04 | WinRate: 100.0% | Loss: 0.3060 | BufferSize: 0 | Solved: True
Epoch 035/400 | Depth: 2 | Steps: 04 | WinRate: 100.0% | Loss: 0.4229 | BufferSize: 0 | Solved: True

[CURRICULUM UPGRADE] Promoted to Complexity Depth Tier: 3!

Epoch 040/400 | Depth: 3 | Steps: 04 | WinRate: 0.0% | Loss: 0.0102 | BufferSize: 0 | Solved: True
Epo

KeyboardInterrupt: 

# **NOTE**: the AI is not good at transforming word problems to equations and stuff.

In [28]:
import re
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

# ==========================================
# 1. LOAD QWEN TRANSLATOR
# ==========================================
QWEN_MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"

llm_tokenizer = AutoTokenizer.from_pretrained(QWEN_MODEL_NAME)
llm_model = AutoModelForCausalLM.from_pretrained(
    QWEN_MODEL_NAME,
    torch_dtype="auto",
    device_map="auto"
)

QWEN_SYSTEM_PROMPT = """You are a mathematical translation assistant for a neural-symbolic solver.
Convert math word problems or equations into an AST expression string matching these format rules:

1. Always output ONLY the expression string wrapped in backticks (e.g., `Eq(x + 5, 10)`).
2. Represent equations using `Eq(LHS, RHS)`.
3. Use simple standard operations: `+`, `-`, `*`, `/`, `**` and other...
4. Use single lowercase letter variables like `x` unless specified.
5. Do NOT include explanations, code blocks, or extra text.
"""

def natural_language_to_ast(text: str) -> ASTNode:
    """Translates natural language/equations via Qwen and parses directly into an ASTNode."""
    messages = [
        {"role": "system", "content": QWEN_SYSTEM_PROMPT},
        {"role": "user", "content": f"Convert to equation string: {text}"}
    ]
    prompt = llm_tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = llm_tokenizer([prompt], return_tensors="pt").to(llm_model.device)

    outputs = llm_model.generate(**inputs, max_new_tokens=48, temperature=0.01, do_sample=False)
    new_tokens = [out[len(inp):] for inp, out in zip(inputs.input_ids, outputs)]
    response = llm_tokenizer.batch_decode(new_tokens, skip_special_tokens=True)[0].strip()

    match = re.search(r'`([^`]+)`', response)
    expr_str = match.group(1) if match else response.strip()
    print(f"Translated AST String: '{expr_str}'")
    return parse_expr(expr_str)

# ==========================================
# 2. RUN SEARCH WITH IN-MEMORY SOLVER
# ==========================================
def solve_prompt(user_prompt: str):
    print(f"\nInput Prompt: \"{user_prompt}\"")
    ast_node = natural_language_to_ast(user_prompt)

    # Uses 'model' directly from your AI.py training session in memory
    path, expansions, status = solve_adaptive(
        model=model,
        start_ast=ast_node,
        task_mode="SOLVE"
    )

    print(f"Solver Status: {status} (Search Expansions: {expansions})")
    if path:
        print("Solution Path:")
        for state, rule in path:
            print(f"  [{rule}] => {state}")

# --- Test Queries ---
solve_prompt("A right-angled triangular solar panel has a hypotenuse of length $c$. If the square of the hypotenuse equals the square of leg $a$ plus the square of leg $b$, solve for hypotenuse $c$ when $a = 3$ and $b = 4$.")
solve_prompt("dad gave me 5 coins, now I have 10, how much I had before?")

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]


Input Prompt: "Find the simplified expression for the change in volume of two cubes, $x$ cubed minus $y$ cubed, divided by the change in their edge lengths, $x$ minus $y$."
Translated AST String: 'Eq((x**3 - y**3) / (x - y), None)'
Solver Status: FAILED (Search Expansions: 1000)

Input Prompt: "dad gave me 5 coins, now I have 10, how much I had before?"
Translated AST String: 'Eq((5 + 10), 15)'
Solver Status: FAILED (Search Expansions: 11)
